# How to track activist investors with Schedule 13D filings

Query new Schedule 13D filings, tell activists from passive 13G holders, and follow one fund across every company it files on.

Read it with charts at https://www.quantisnow.com/sec-api/guides/track-activist-13d-filings. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Anyone who ends up owning more than 5% of a public company's stock has to say so. Passive holders like index funds use the short Schedule 13G. Anyone who might push the company, ask for board seats or a sale files the longer 13D, due within five business days of crossing 5%. That makes a new 13D one of the earliest public signs that an activist is in a stock.

## Every new 13D this month

### Which activist 13D stakes were filed in the last 30 days?

A 13D is due within five business days of crossing 5%. The list is short enough to read every week.

```shell
curl -X POST https://www.quantisnow.com/api/v1/ownership \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "formType:13D AND filedAt:[2026-10-25 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"filedAt": {"order": "desc"}}]
}'
```

Response (real, first 2 of 56):

```json
{
  "total": {
    "value": 56,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001418812-26-000003",
      "accessionNo": "0001418812-26-000003",
      "formType": "13D",
      "filedAt": "2026-11-23T18:36:03-05:00",
      "filer": {
        "cik": "0001418812",
        "name": "VA Partners I, LLC"
      },
      "subject": {
        "cik": "0001576789",
        "name": "Wix.com Ltd.",
        "ticker": "WIX"
      },
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1576789/000141881226000003/0001418812-26-000003-index.htm"
    },
    {
      "id": "0002141359-26-000002",
      "accessionNo": "0002141359-26-000002",
      "formType": "13D",
      "filedAt": "2026-11-23T16:11:08-05:00",
      "filer": {
        "cik": "0002141359",
        "name": "Wall Douglas Scott"
      },
      "subject": {
        "cik": "0001903595",
        "name": "Brag House Holdings Inc.",
        "ticker": "TBH"
      },
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1903595/000214135926000002/0002141359-26-000002-index.htm"
    }
  ]
}
```

The list is short, usually a few dozen a month, which makes it worth reading in full each week. Join it with the [8-K events endpoint](https://www.quantisnow.com/sec-api/8-k-events) on the same ticker to see what the company said back: a board change (item 5.02), a new agreement (1.01) or a rights plan (3.03).

## Follow one filer

Search by the filer's name with a trailing wildcard to catch every entity a manager files under, then switch to `filer.cik` once you know which one you want.

### Which companies has BlackRock reported a 5% stake in recently?

A large passive holder's recent 5% filings, across every company it reports on.

```shell
curl -X POST https://www.quantisnow.com/api/v1/ownership \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "filer.name:BlackRock* AND formType:13G AND filedAt:[2026-08-26 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"filedAt": {"order": "desc"}}]
}'
```

Response (real, first 2 of 46):

```json
{
  "total": {
    "value": 46,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001493152-26-052434",
      "accessionNo": "0001493152-26-052434",
      "formType": "13G",
      "filedAt": "2026-11-16T16:40:56-05:00",
      "filer": {
        "cik": "0002052113",
        "name": "BlackRock Portfolio Management LLC"
      },
      "subject": {
        "cik": "0001515324",
        "name": "Ares Dynamic Credit Allocation Fund Inc.",
        "ticker": "ARDC"
      },
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1515324/000149315226052434/0001493152-26-052434-index.htm"
    },
    {
      "id": "0002012383-26-003972",
      "accessionNo": "0002012383-26-003972",
      "formType": "13G",
      "filedAt": "2026-11-04T16:05:25-05:00",
      "filer": {
        "cik": "0002012383",
        "name": "BlackRock, Inc."
      },
      "subject": {
        "cik": "0002152623",
        "name": "Navient Corporation",
        "ticker": "JSM"
      },
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/2152623/000201238326003972/0002012383-26-003972-index.htm"
    }
  ]
}
```

In [ ]:
import os, requests

res = requests.post(
    "https://www.quantisnow.com/api/v1/ownership",
    headers={"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"},
    json={"query": "filer.name:Elliott* AND formType:13D", "size": 50},
)
for r in res.json()["data"]:
    print(r["filedAt"][:10], r["subject"]["ticker"], r["subject"]["name"])